# Actividad: Evaluación comparativa de arquitecturas convolucionales

Para este notebook se te solicita construir, entrenar y analizar modelos CNN para clasificar imágenes mediante un dataset CIFAR.

**Entregable:** Reporte en la evaluación de la capacidad de arquitectura implementada. Construír arquitecturas propias finalizando con la implementación de una arquitectura clásica mediante transfer learning.


## Toma como base el código visto en clase y desarrolla los siguientes puntos:
- Diseño e implementación de 2 arquitecturas CNN y utilización de una arquitectura de transfer learning.

- Buen uso de data augmentation y regularización.

- Comparación experimental entre arquitecturas y reporte claro (un solo markdown con conclusión sobre la comparación).





In [ ]:
import os
from pathlib import Path

os.environ["MPLCONFIGDIR"] = str(Path.cwd() / ".matplotlib_cache")
os.environ["KERAS_HOME"] = str(Path.cwd() / ".keras_cache")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU disponible:", bool(tf.config.list_physical_devices("GPU")))


## Definiciones de modelos

In [ ]:
# =============================
# Carga y preparación de datos
# =============================
# CIFAR-10 contiene 60,000 imágenes RGB de 32x32 en 10 clases.
# Para que la práctica sea ejecutable en una computadora común, se usa un subconjunto.
# Puedes poner TRAIN_SAMPLES = None y TEST_SAMPLES = None para usar todo el dataset.

CLASS_NAMES = [
    "avion", "auto", "ave", "gato", "venado",
    "perro", "rana", "caballo", "barco", "camion"
]

TRAIN_SAMPLES = 20000
VAL_SAMPLES = 5000
TEST_SAMPLES = 5000
BATCH_SIZE = 64
EPOCHS = 8

def load_cifar10_dataset():
    """Carga CIFAR-10 desde cache local; descarga solo si no existe."""
    import pickle

    dataset_dirs = [
        Path.cwd() / ".keras_cache" / "datasets" / "cifar-10-batches-py",
        Path.home() / ".keras" / "datasets" / "cifar-10-batches-py",
        Path.home() / ".keras" / "datasets" / "cifar-10-batches-py-target" / "cifar-10-batches-py",
    ]

    dataset_dir = next((folder for folder in dataset_dirs if (folder / "data_batch_1").exists()), None)
    if dataset_dir is None:
        print("CIFAR-10 no esta en cache; descargando a .keras_cache/datasets...")
        dataset_dir = Path(
            keras.utils.get_file(
                "cifar-10-batches-py",
                origin="https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz",
                untar=True,
                file_hash="6d958be074577803d12ecdefd02955f39262c83c16fe9348329d7fe0b5c001ce",
                cache_dir=str(Path.cwd() / ".keras_cache"),
            )
        )

    def load_batch(batch_path):
        with open(batch_path, "rb") as file:
            batch = pickle.load(file, encoding="latin1")
        images = batch["data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        labels = np.array(batch["labels"], dtype="uint8")
        return images, labels

    train_images = []
    train_labels = []
    for batch_number in range(1, 6):
        images, labels = load_batch(dataset_dir / f"data_batch_{batch_number}")
        train_images.append(images)
        train_labels.append(labels)

    x_train_loaded = np.concatenate(train_images, axis=0)
    y_train_loaded = np.concatenate(train_labels, axis=0)
    x_test_loaded, y_test_loaded = load_batch(dataset_dir / "test_batch")
    return (x_train_loaded, y_train_loaded), (x_test_loaded, y_test_loaded)


(x_train_full, y_train_full), (x_test_full, y_test_full) = load_cifar10_dataset()
y_train_full = y_train_full.reshape(-1)
y_test_full = y_test_full.reshape(-1)

if TRAIN_SAMPLES is not None:
    x_train_full = x_train_full[:TRAIN_SAMPLES + VAL_SAMPLES]
    y_train_full = y_train_full[:TRAIN_SAMPLES + VAL_SAMPLES]
if TEST_SAMPLES is not None:
    x_test_full = x_test_full[:TEST_SAMPLES]
    y_test_full = y_test_full[:TEST_SAMPLES]

x_val = x_train_full[:VAL_SAMPLES].astype("float32") / 255.0
y_val = y_train_full[:VAL_SAMPLES]
x_train = x_train_full[VAL_SAMPLES:].astype("float32") / 255.0
y_train = y_train_full[VAL_SAMPLES:]
x_test = x_test_full.astype("float32") / 255.0
y_test = y_test_full

print("Train:", x_train.shape, y_train.shape)
print("Validacion:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(CLASS_NAMES[y_train[i]])
    plt.axis("off")
plt.suptitle("Ejemplos de CIFAR-10")
plt.tight_layout()
plt.show()

# =============================
# Data augmentation y modelos
# =============================
# La regularizacion se aplica con Dropout, BatchNormalization, L2 y EarlyStopping.

data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal", seed=SEED),
        layers.RandomRotation(0.08, seed=SEED),
        layers.RandomZoom(0.10, seed=SEED),
        layers.RandomTranslation(0.08, 0.08, seed=SEED),
        layers.RandomContrast(0.10, seed=SEED),
    ],
    name="data_augmentation",
)


def build_cnn_simple(input_shape=(32, 32, 3), num_classes=10):
    """CNN propia 1: arquitectura compacta para establecer una linea base."""
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="CNN_simple")


def build_cnn_regularizada(input_shape=(32, 32, 3), num_classes=10):
    """CNN propia 2: mas profunda, con BatchNorm, Dropout y regularizacion L2."""
    weight_decay = 1e-4
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)

    for filters in [32, 64, 128]:
        x = layers.Conv2D(
            filters, 3, padding="same", use_bias=False,
            kernel_regularizer=regularizers.l2(weight_decay),
        )(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.Conv2D(
            filters, 3, padding="same", use_bias=False,
            kernel_regularizer=regularizers.l2(weight_decay),
        )(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.20)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(weight_decay))(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="CNN_regularizada")


def build_transfer_mobilenet(input_shape=(32, 32, 3), num_classes=10):
    """Transfer learning con MobileNetV2 preentrenada en ImageNet."""
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = layers.Resizing(96, 96)(x)
    x = layers.Lambda(lambda image: preprocess_input(image * 255.0), name="mobilenet_preprocess")(x)

    try:
        base_model = MobileNetV2(
            include_top=False,
            weights="imagenet",
            input_shape=(96, 96, 3),
        )
        print("MobileNetV2 cargada con pesos de ImageNet.")
    except Exception as exc:
        print("No se pudieron descargar/cargar pesos ImageNet. Se usaran pesos aleatorios.")
        print("Detalle:", exc)
        base_model = MobileNetV2(
            include_top=False,
            weights=None,
            input_shape=(96, 96, 3),
        )

    base_model.trainable = False
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="MobileNetV2_transfer")


models = {
    "CNN simple": build_cnn_simple(),
    "CNN regularizada": build_cnn_regularizada(),
    "MobileNetV2 transfer": build_transfer_mobilenet(),
}

for name, model in models.items():
    print("\n", "=" * 80)
    print(name)
    model.summary()


## Entrenamiento de modelos.

In [ ]:
# =============================
# Compilacion y entrenamiento
# =============================

def compile_model(model, learning_rate=1e-3):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1,
    ),
]

histories = {}
results = []

for name, model in models.items():
    print("\n" + "=" * 80)
    print(f"Entrenando: {name}")

    lr = 3e-4 if "transfer" in name.lower() else 1e-3
    compile_model(model, learning_rate=lr)

    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1,
    )
    histories[name] = history

    test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
    best_val_accuracy = max(history.history["val_accuracy"])
    results.append(
        {
            "modelo": name,
            "best_val_accuracy": best_val_accuracy,
            "test_accuracy": test_accuracy,
            "test_loss": test_loss,
            "epochs_entrenadas": len(history.history["loss"]),
            "parametros": model.count_params(),
        }
    )

results_df = pd.DataFrame(results).sort_values("test_accuracy", ascending=False)
results_df


## Estadística y gráficos

In [ ]:
# =============================
# Estadistica, graficos y evaluacion
# =============================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, history in histories.items():
    axes[0].plot(history.history["accuracy"], marker="o", label=f"{name} train")
    axes[0].plot(history.history["val_accuracy"], marker="o", linestyle="--", label=f"{name} val")
    axes[1].plot(history.history["loss"], marker="o", label=f"{name} train")
    axes[1].plot(history.history["val_loss"], marker="o", linestyle="--", label=f"{name} val")

axes[0].set_title("Accuracy por epoca")
axes[0].set_xlabel("Epoca")
axes[0].set_ylabel("Accuracy")
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=8)

axes[1].set_title("Loss por epoca")
axes[1].set_xlabel("Epoca")
axes[1].set_ylabel("Loss")
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
plt.bar(results_df["modelo"], results_df["test_accuracy"])
plt.title("Comparacion de accuracy en test")
plt.ylabel("Test accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

best_model_name = results_df.iloc[0]["modelo"]
best_model = models[best_model_name]
print("Mejor modelo:", best_model_name)
display(results_df)

y_pred_proba = best_model.predict(x_test, batch_size=BATCH_SIZE, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

print("Reporte de clasificacion del mejor modelo:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 7))
plt.imshow(cm, cmap="Blues")
plt.title(f"Matriz de confusion - {best_model_name}")
plt.colorbar()
plt.xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha="right")
plt.yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
plt.xlabel("Prediccion")
plt.ylabel("Etiqueta real")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
plt.tight_layout()
plt.show()

# Ejemplos de predicciones
sample_idx = np.random.default_rng(SEED).choice(len(x_test), size=12, replace=False)
plt.figure(figsize=(12, 6))
for plot_i, idx in enumerate(sample_idx, start=1):
    plt.subplot(3, 4, plot_i)
    plt.imshow(x_test[idx])
    true_label = CLASS_NAMES[y_test[idx]]
    pred_label = CLASS_NAMES[y_pred[idx]]
    color = "green" if y_test[idx] == y_pred[idx] else "red"
    plt.title(f"Real: {true_label}\nPred: {pred_label}", color=color, fontsize=9)
    plt.axis("off")
plt.suptitle(f"Predicciones de {best_model_name}")
plt.tight_layout()
plt.show()


# Conclusiones

En esta práctica se compararon tres enfoques para clasificar imágenes de CIFAR-10: una CNN simple, una CNN regularizada y una arquitectura clásica mediante transfer learning con MobileNetV2. La CNN simple funciona como línea base porque tiene pocas capas y pocos mecanismos de regularización; por eso suele entrenar rápido, pero puede quedarse corta para capturar patrones visuales más complejos. La CNN regularizada agrega Batch Normalization, Dropout y penalización L2, lo que normalmente mejora la generalización y reduce el sobreajuste frente a la CNN simple.

El mejor modelo debe identificarse con la tabla `results_df`, usando principalmente `test_accuracy` y también revisando la diferencia entre accuracy de entrenamiento y validación. Si MobileNetV2 obtiene el mejor resultado, la razón probable es que sus filtros preentrenados en ImageNet ya reconocen bordes, texturas y formas útiles, aunque CIFAR-10 tenga imágenes pequeñas. Si la CNN regularizada supera a MobileNetV2, la explicación probable es que el subconjunto de entrenamiento, el tamaño 32x32 o pocas épocas limitaron el beneficio del transfer learning.

Para mejorar el experimento, aumentaría `TRAIN_SAMPLES`, entrenaría por más épocas, haría fine-tuning desbloqueando las últimas capas de MobileNetV2 con una tasa de aprendizaje baja y probaría más aumentos de datos. También convendría repetir cada entrenamiento varias veces o usar validación cruzada para reducir el efecto de la aleatoriedad y reportar resultados más robustos.
